![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 1. Parte A: De texto a tensor

**Asignatura:** Modelos de Lenguaje
**Bloque:** 1 — Introducción a las Redes Neuronales

---

## Introducción

Una red neuronal no opera sobre texto: opera sobre tensores. Las operaciones que la componen —productos entre matrices, no linealidades, derivadas— están definidas sobre números en punto flotante, y no hay ninguna de ellas que reciba una cadena de caracteres. Por eso, antes de que exista cualquier modelo, hace falta convertir el texto en números, y hacerlo de una manera que conserve lo que importa.

El corpus de este laboratorio son órdenes dirigidas a un asistente por voz, del tipo *"apagá la luz de la cocina"*, y la tarea es clasificar cada una según el dominio al que pertenece. Para que una de esas frases llegue a la primera capa de un modelo alguien tuvo que decidir en qué unidades se corta el texto —¿palabras?, ¿letras?, ¿pedazos de palabra?—, cuáles de esas unidades entran al vocabulario y cuáles quedan afuera, qué se pone en lugar de las que quedaron afuera, y cómo se acomodan en una matriz rectangular varias frases que tienen largos distintos. Ninguna de esas decisiones es obvia, y todas dejan huella: la información que se pierde acá el modelo no la recupera después, por mucho que se entrene.

Esa cadena de decisiones es el tema de esta primera parte. No vas a entrenar nada todavía: vas a construir la maquinaria que convierte texto en el tensor `(B, L)` que la Parte B va a consumir, y a medir el efecto de cada decisión que tomes en el camino.

Al completar este laboratorio vas a poder:

- Crear, transformar y consultar tensores de PyTorch, y anticipar cuándo dos tensores comparten memoria.
- Usar *broadcasting* para aplicar una máscara sobre un lote de vectores.
- Calcular gradientes con `autograd` y verificarlos contra la derivada analítica.
- Escribir tokenizadores y medir sus efectos sobre un corpus real en español.
- Implementar una clase `Vocabulario` completa, y elegir su umbral de frecuencia con datos y no a ojo.
- Truncar, rellenar y enmascarar un lote de textos, y cuantificar cuánto del tensor resultante es relleno.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- Para resolver cada ejercicio, consultá el material teórico de las Clases 1 y 2 de la Unidad 1.
- **Este laboratorio corre entero en CPU.** No hace falta GPU, y activarla no lo va a hacer más rápido.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas —enunciados, explicaciones, ejemplos provistos y encabezado— **no se tocan**.

La corrección se hace celda por celda: cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo suelto, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las dos celdas que siguen ya vienen resueltas. No hay nada que completar en ellas, pero **hay que ejecutarlas** antes de empezar, y conviene leerlas porque definen los nombres que usan todos los ejercicios.

La primera importa las librerías. La segunda descarga el corpus y deja en memoria tres `DataFrame` de pandas —`train`, `val` y `test`— y la lista `ESCENARIOS`, que traduce el índice de cada clase al nombre del dominio.

In [ ]:
# ─── Setup: imports ─────────────────────────────────────────────────────────
# Todo esto viene preinstalado en Colab: no hace falta instalar nada.
import re
import math
import collections
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

print(f"Versión de PyTorch: {torch.__version__}")
print(f"Versión de pandas:  {pd.__version__}")

### El corpus: MASSIVE en español

MASSIVE es un corpus de órdenes a un asistente de voz que Amazon recolectó en 51 idiomas. Nosotros usamos la partición en español de España (`es-ES`) de la variante *scenario*, en la que cada orden viene etiquetada con el dominio al que pertenece: `alarm`, `iot`, `calendar`, `weather` y quince más.

Son órdenes escritas por hablantes nativos, con sus tipeos y sus acentos faltantes. Esa suciedad no es un defecto del corpus, es justamente lo que vas a medir en la Sección B.

Los tres *splits* —entrenamiento, validación y prueba— ya vienen separados por los autores del corpus y entre los tres pesan menos de 2 MB. La celda los lee de una copia guardada en el repositorio de la materia, y si ese servidor no responde los busca en el Hub de Hugging Face. Es el mismo archivo en los dos lados; tener dos fuentes evita que la clase dependa de una sola.

Además de cargar los datos, la celda imprime un resumen: cuántas órdenes tiene cada *split*, las primeras seis filas y la distribución de largos en palabras. Miralo antes de seguir, porque el largo típico de una orden es el número con el que vas a decidir `L` en el Ejercicio 7.

In [ ]:
# ─── Setup: el corpus MASSIVE en español ────────────────────────────────────
REPO = "https://github.com/javovelez/Modelos-de-Lenguaje/raw/main/datos"
HUB  = "https://huggingface.co/datasets/SetFit/amazon_massive_scenario_es-ES/resolve/main"


def leer_split(nombre):
    """Lee un split del corpus, del repo de la materia o del Hub como respaldo."""
    try:
        return pd.read_json(f"{REPO}/{nombre}.jsonl", lines=True)
    except Exception:
        return pd.read_json(f"{HUB}/{nombre}.jsonl", lines=True)


train = leer_split("train")
val   = leer_split("validation")
test  = leer_split("test")

# `label` es el índice numérico del escenario; `label_text`, su nombre.
# Esta lista traduce índice -> nombre y la vamos a reusar en la Parte B.
ESCENARIOS = (train[["label", "label_text"]]
              .drop_duplicates()
              .sort_values("label")["label_text"]
              .tolist())

print(f"entrenamiento: {len(train):>6,} órdenes")
print(f"validación:    {len(val):>6,} órdenes")
print(f"prueba:        {len(test):>6,} órdenes")
print(f"escenarios:    {len(ESCENARIOS)}")
print()
print(train.head(6).to_string(index=False))

largos = train.text.str.split().str.len()
print()
print(f"palabras por orden: media {largos.mean():.1f}, mediana {largos.median():.0f}, "
      f"percentil 95 {np.percentile(largos, 95):.0f}, máximo {largos.max()}")

---
## Sección A: Mecánica de tensores

Los primeros cuatro ejercicios son de mecánica pura, con tensores inventados. Están descontextualizados a propósito: el objetivo es que las operaciones te salgan sin pensarlas, porque a partir de la Sección B van a aparecer todas juntas y con texto adentro.

Prestá atención a los ejercicios 2 y 3 en particular. El 2 tiene una trampa que causa bugs muy difíciles de encontrar, y el 3 es exactamente la operación que hace el clasificador que vas a construir en la Parte B.

### Ejercicio 1 — Creación, forma y tipo de un tensor

**Objetivo:** Crear tensores de las tres maneras que vas a usar todo el curso y leer sus atributos de forma y tipo.

**Enunciado:**

1. **A partir de datos predefinidos.** Creá un tensor y guardalo en una variable llamada `ids`, con estos dos renglones de números:

   ```
   5  12  7  0  0
   9   3  0  0  0
   ```

   Tiene que ser de tipo `torch.long`, el tipo entero de 64 bits que PyTorch usa para índices. `ids` imita un lote de dos frases ya codificadas: cada fila es una frase, cada número es la posición de una palabra en el vocabulario, y los ceros del final son relleno.

2. **A partir de una forma, sin datos.** Creá un tensor lleno de ceros, de 4 filas por 6 columnas y también de tipo `torch.long`, y guardalo en una variable llamada `ceros`.

3. **Con números al azar, pero reproducibles.** Fijá la semilla del generador de PyTorch en `0` y creá, en una variable llamada `vectores`, un tensor de forma `(4, 6, 8)` con números al azar entre 0 y 1. Imprimí su suma total redondeada a cuatro decimales: si fijaste la semilla justo antes de crearlo, tiene que darte `92.1386`.

4. **Leé los atributos de los tres.** Para cada uno de los tres tensores imprimí, en una línea, su forma, su tipo de dato, su cantidad de dimensiones y su cantidad total de elementos.

5. **Provocá un error de tipo a propósito.** Creá una capa `torch.nn.Embedding` para un vocabulario de 10 palabras y vectores de 4 dimensiones —una tabla de 10 filas, un vector por palabra— y buscá en ella la fila de índice 3 dos veces: con un tensor que contenga el entero `3`, imprimiendo la forma de lo que devuelve, y con uno que contenga `3.0`. La segunda búsqueda falla, con el mismo número. Envolvela en un `try/except` para que el error no corte la ejecución del notebook, e imprimí el nombre de la clase de la excepción y su mensaje.

> **Pista:** Los cuatro atributos del punto 4 son `.shape`, `.dtype`, `.ndim` y `.numel()`. El argumento `dtype` está disponible en todos los constructores (`torch.tensor`, `torch.zeros`, `torch.ones`). Para el punto 5 alcanza con `except Exception as e:`, y el nombre de la clase de la excepción sale de `type(e).__name__`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Los tensores `ids` y `vectores` quedaron con tipos de dato distintos: `torch.long` el primero, `torch.float32` el segundo. ¿Por qué un tensor de índices de vocabulario **no puede** ser de punto flotante, mientras que un tensor de *embeddings* **tiene que** serlo?

*(Escribí tu respuesta acá)*

### Ejercicio 2 — Cambiar la forma, y la trampa de la memoria compartida

**Objetivo:** Cambiar la forma de un tensor con `reshape`, `unsqueeze` y `squeeze`, y reconocer cuándo el resultado comparte memoria con el original.

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Enunciado — Parte A: cambiar la forma.**

Creá un tensor `t` con los números enteros del 0 al 23 acomodados en la forma `(2, 3, 4)`. Después imprimí la forma que devuelve cada una de estas transformaciones:

1. Reacomodarlo en 6 filas de 4 columnas.
2. Aplanarlo a un vector de una sola dimensión, sin escribir vos el número 24: dejá que PyTorch deduzca esa dimensión.
3. Agregarle una dimensión de tamaño 1 al final, de modo que la forma pase de `(2, 3, 4)` a `(2, 3, 4, 1)`.
4. Quitarle esa dimensión que agregaste, y verificar con una comparación que volviste a la forma original.

> **Pista:** Las cuatro transformaciones están en la Clase 1, en la parte de cambiar la forma de un tensor. Ninguna copia datos ni cambia el contenido: lo único que cambia es cómo se recorre el mismo bloque de números, y por eso el total de elementos tiene que dar siempre 24.

In [ ]:
# Tu código aquí

**Enunciado — Parte B: la trampa.**

Cuando puede, `reshape` no copia nada: devuelve una **vista**, un tensor nuevo que mira el mismo bloque de memoria que el original. Escribir en la vista es escribir en el original, y eso es fuente de bugs difíciles. Comprobalo:

1. Creá un tensor `a` con los números enteros del 0 al 5 e imprimilo.
2. Creá `b` reacomodando `a` en 2 filas de 3 columnas. Escribí un `99` en la posición `[0, 0]` de `b` y volvé a imprimir `a`.
3. Creá `c` reacomodando una **copia** de `a` en 2 filas de 3 columnas. Escribí un `-1` en la posición `[0, 1]` de `c`, e imprimí `c` y `a`.
4. Cerrá comprobando si `b` y `c` arrancan en la misma dirección de memoria que `a`.

> **Pista 1:** La manera de pedir una copia de verdad está en la Clase 1, en la parte de modificar tensores en su lugar.

> **Pista 2:** Para el punto 4 usá el método `.data_ptr()`, que devuelve la dirección de memoria donde arrancan los datos de un tensor. No aparece en la teoría: lo agregamos acá porque es la forma más directa de comprobar si dos tensores comparten memoria, en lugar de deducirlo mirando qué valores cambiaron.

> **Pista 3:** No expliques nada en el código todavía; eso va en la pregunta de análisis. Limitate a mostrar los valores.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Hay una técnica para que un clasificador de texto aprenda a manejar palabras que nunca vio: durante el entrenamiento —y solo durante el entrenamiento— se reemplaza al azar una fracción chica de los tokens de cada frase por `<unk>`. Así la fila de `<unk>` recibe gradiente y el modelo no queda dependiendo demasiado de cada palabra puntual. Se la conoce como *word dropout*.

Alguien la implementa así, adentro de un `Dataset` cuyo atributo `self.X` es el tensor `(N, L)` con el corpus entero ya codificado:

```python
def __getitem__(self, i):
    fila = self.X[i]                      # la fila i del corpus
    sorteo = torch.rand(len(fila)) < 0.1  # el 10% de las posiciones, al azar
    fila[sorteo] = self.unk_id            # esas van a <unk>
    return fila, self.y[i]
```

a) Explicá qué le pasa a `self.X` a medida que el `DataLoader` recorre el corpus época tras época, y por qué este bug es especialmente difícil de encontrar.

b) ¿Cómo lo arreglarías?

*(Escribí tu respuesta acá)*

### Ejercicio 3 — Broadcasting de una máscara sobre un lote de vectores

**Objetivo:** Aplicar una máscara `(B, L, 1)` sobre un lote de vectores `(B, L, E)` usando *broadcasting*, y calcular un promedio que ignore el relleno.

**Enunciado:**

Este ejercicio es la operación central del clasificador que vas a construir en la Parte B, aislada y con números chicos para que puedas verificarla a mano.

Partí de este lote de índices, donde el `0` representa relleno:

```python
lote = torch.tensor([[5, 12, 7, 0, 0],
                     [9,  3, 0, 0, 0]])
```

Una capa de *embeddings* reemplaza cada uno de esos índices por el vector de la palabra que le toca —la misma búsqueda del Ejercicio 1, pero para todo el lote de una vez—, así que el `(2, 5)` de `lote` se convierte en un `(2, 5, 4)`: en la posición `[i, j]` queda el vector de 4 dimensiones de la palabra `lote[i, j]`. Ese tensor, y no `lote`, es sobre el que opera el modelo. Como todavía no tenés una tabla entrenada de donde sacarlo, lo vas a simular con ruido: para enmascarar y promediar, lo único que importa es la forma.

1. **Simulá los vectores.** Fijá la semilla del generador de PyTorch en `0` y creá, en una variable `vecs`, un tensor de forma `(2, 5, 4)` con números al azar de una normal estándar (media 0, desvío 1).

2. **Construí la máscara con la forma correcta.** A partir de `lote`, armá una variable `mascara` de forma `(2, 5, 1)` y tipo flotante, que valga 1 en las posiciones con token real y 0 en las de relleno. Imprimí su forma.

3. **Aplicá la máscara** multiplicando los vectores por ella. Imprimí la forma del resultado y verificá, con una comparación que imprima `True` o `False`, que las posiciones de relleno de la primera frase quedaron efectivamente en cero.

4. **Calculá el promedio enmascarado** en una variable `promedio`: sumá sobre la dimensión de la secuencia y dividí por la cantidad de tokens **reales** de cada frase, en lugar de dividir por las 5 posiciones. Cuidá que el denominador nunca pueda ser cero.

5. **Calculá el promedio ingenuo** en una variable `ingenuo`: el promedio sobre la dimensión de la secuencia sin mirar la máscara, que divide siempre por 5. Imprimí los dos resultados de la primera frase, uno debajo del otro, para poder compararlos.

6. **Verificá que tu promedio enmascarado es el correcto.** Calculá aparte el promedio de las 3 filas reales de la primera frase y compará ese vector con el que te dio el promedio enmascarado usando `torch.allclose`, que devuelve `True` si dos tensores coinciden hasta la tolerancia numérica. Compará también contra el ingenuo, para ver que no coincide.

> **Pista 1:** El *broadcasting* estira automáticamente las dimensiones de tamaño 1. Al multiplicar `(2, 5, 4)` por `(2, 5, 1)`, la última dimensión de la máscara se replica 4 veces: el mismo 0 o 1 se aplica a las 4 componentes del vector de esa posición. Por eso la máscara necesita ese eje extra: con forma `(2, 5)` contra `(2, 5, 4)` las dimensiones no alinean.

> **Pista 2:** La cantidad de tokens reales por frase es la suma de la máscara sobre la dimensión de la secuencia, y sale ya con la forma `(2, 1)` que hace falta para dividir. Para blindar esa división, la Clase 1 usa `.clamp(min=1)`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

En este ejercicio los vectores de las posiciones de relleno eran ruido (`torch.randn` los llenó igual que a los demás). En el modelo real, `nn.Embedding(..., padding_idx=0)` garantiza que la fila del `<pad>` sea **todo ceros**.

Con esa garantía, ¿el promedio ingenuo `vecs.mean(dim=1)` deja de estar mal? Justificá, y explicá qué relación exacta habría entre el promedio ingenuo y el enmascarado en ese caso.

*(Escribí tu respuesta acá)*

### Ejercicio 4 — Autograd: verificar un gradiente y controlar su acumulación

**Objetivo:** Calcular gradientes con `autograd`, verificarlos contra la derivada analítica, y entender por qué hay que descartarlos en cada paso.

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Enunciado — Parte A: verificación analítica.**

Considerá la función $y = w^2 x$.

1. Derivá $\partial y / \partial w$ a mano. Es una línea: escribila como comentario en el código, junto con el valor que toma para $w = 2$ y $x = 3$.
2. Creá dos tensores de un solo elemento: uno llamado `w` con el valor `2.0`, marcado para que PyTorch registre las operaciones en las que participa, y otro llamado `x` con el valor `3.0`, sin marcar.
3. Calculá `y` con la fórmula de arriba e imprimí su valor junto a su atributo `.grad_fn`, que es el rastro que dejó *autograd*: dice de qué operación salió `y`, y es el grafo que después se recorre hacia atrás.
4. Pedile a PyTorch que propague el gradiente hacia atrás desde `y`.
5. Imprimí el gradiente que quedó acumulado en `w` junto al valor analítico del punto 1, y verificá con una comparación que coinciden.

> **Pista:** Los gradientes no se calculan solos: hay que pedirlos, y solo se guardan en los tensores que están marcados para eso. Las dos cosas —cómo se marca un tensor y cómo se dispara la propagación— están en la Clase 2.

In [ ]:
# Tu código aquí

**Enunciado — Parte B: la acumulación.**

1. Creá un tensor `w2`, otra vez con el valor `2.0` y otra vez marcado para que se le calcule el gradiente.
2. Adentro de un bucle de tres iteraciones, recalculá `y` con la misma fórmula de la Parte A, propagá el gradiente hacia atrás e imprimí en cada vuelta el gradiente acumulado en `w2`. La derivada vale lo mismo en las tres vueltas: mirá qué pasa igual con el valor impreso.
3. Descartá el gradiente acumulado, repetí el cálculo una vez más e imprimí el resultado.

> **Pista 1:** Propagar hacia atrás **suma** el gradiente nuevo al que ya estaba guardado en `.grad`, en lugar de reemplazarlo. Es una decisión deliberada de PyTorch, no un descuido: pensá para qué puede servir acumular.

> **Pista 2:** Para descartar el gradiente de un tensor suelto alcanza con asignarle `None` a su atributo `.grad`. En un modelo entero eso mismo lo hace `zero_grad()`, que recorre todos los parámetros.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Si en un loop de entrenamiento te olvidás de llamar a `zero_grad()`, ¿qué le pasa al tamaño de los pasos de actualización a medida que avanzan las épocas? ¿Por qué el efecto se parece al de una tasa de aprendizaje mal elegida?

b) La acumulación es deliberada. Describí una situación de entrenamiento en la que **querés** que los gradientes se sumen antes de actualizar.

*(Escribí tu respuesta acá)*

---
## Sección B: De texto a tensor

Acá empieza el trabajo con el corpus real. La cadena tiene cuatro eslabones y los vas a construir en orden: **tokenizar** (cortar el texto en unidades), **armar el vocabulario** (decidir qué unidades existen y qué se hace con las demás), **codificar** (reemplazar cada unidad por su índice) y **darle forma rectangular** (truncar, rellenar y enmascarar).

Cada eslabón tiene un parámetro que parece menor y no lo es. La idea de esta sección es que ninguno lo elijas a ojo: en cada caso vas a medir el efecto antes de decidir.

### Ejercicio 5 — Dos tokenizadores, y qué le hace cada uno al corpus

**Objetivo:** Escribir dos tokenizadores con criterios distintos, observar su comportamiento sobre casos difíciles y medir la diferencia sobre el corpus completo.

**Enunciado:**

1. **Escribí `tok_simple(texto)`**: pasa el texto a minúsculas y devuelve todas sus secuencias alfanuméricas. Es una sola línea, con una expresión regular.

2. **Escribí `tok_sin_acentos(texto)`**: hace lo mismo que el anterior, pero además elimina los acentos, de manera que `envía` y `envia` produzcan el mismo token. La receta estándar tiene dos pasos: normalizar el texto a una forma que separe cada letra acentuada en letra base más marca diacrítica, y después descartar las marcas. El módulo `unicodedata` resuelve los dos.

3. **Probá los dos sobre esta lista de casos difíciles** e imprimí el resultado de cada uno. Son **órdenes reales del corpus**, elegidas porque cada una rompe algo distinto:

```python
dificiles = [
    "cuáles son las noticias en t. v. e. noticias",   # una sigla, partida en letras
    "cual es ese álbum de música actual",             # 'cual' sin tilde
    "cuales son las ultimas noticias",                # dos palabras sin tilde
    "envía un correo electronico a raul",             # tipeos sin tilde
]
```

4. **Medí los dos sobre el corpus de entrenamiento completo.** Para cada tokenizador, contá cuántos tokens totales y cuántos tokens **distintos** produce sobre `train.text`, e imprimí una línea por tokenizador. Guardate los dos contadores, que los vas a necesitar en el punto siguiente.

5. **Mostrá qué se fusionó.** Encontrá los grupos de palabras del corpus que colapsan en la misma forma al sacarles los acentos (por ejemplo `qué` y `que`), contá cuántos grupos hay e imprimí los cinco más frecuentes, con las variantes de cada uno y su cantidad de apariciones.

> **Pista 1:** La clase `Counter` de `collections` cuenta las apariciones de un iterable y responde las dos preguntas del punto 4 de una vez: el total es la suma de sus valores y las formas distintas son su largo. En la expresión regular, `\w` incluye en Python 3 las letras acentuadas y la `ñ`.

> **Pista 2:** Para el punto 5, agrupá las palabras del vocabulario por su versión sin acentos usando un `collections.defaultdict(list)`, y quedate con los grupos que tengan más de un elemento.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Mirá los grupos que colapsan. Algunos son claramente un error de tipeo que conviene unificar (`electrónico` / `electronico`), pero otros son **palabras distintas del español** que quedan fusionadas.

a) Identificá al menos dos grupos del segundo tipo y explicá qué información se pierde al fusionarlos.

b) Para este corpus en particular —órdenes cortas a un asistente de voz, escritas por gente que no siempre pone los acentos— ¿qué tokenizador elegirías y por qué? Justificá con los números que obtuviste, no con una preferencia general.

*(Escribí tu respuesta acá)*

### Ejercicio 6 — La clase `Vocabulario`

**Objetivo:** Implementar la clase que mapea tokens a índices y viceversa, medir el efecto del umbral de frecuencia, y elegirlo con datos.

**Enunciado:**

Es el ejercicio grande de esta parte. La clase que escribas acá la vas a volver a usar en la Parte B y en el Laboratorio 2, así que vale la pena que quede prolija. Son tres partes y cada una va en su propia celda de código.

Es también la clase que la Clase 1 construye en la sección *2.4 Una clase `Vocabulario`*. Acá la escribís vos, y con una interfaz fija: los laboratorios que siguen la van a usar tal cual, así que los nombres de los métodos y de los atributos no son negociables.

**Enunciado — Parte A: la clase.** Implementá `Vocabulario` con esta interfaz:

- `__init__(self, textos, tokenizador=tok_simple, freq_min=1, max_tokens=None)`. Cuenta todos los tokens de `textos` en `self.contador`, se queda con los que aparecen al menos `freq_min` veces (y con los `max_tokens` más frecuentes, si se especifica) y arma las dos estructuras del mapeo: `self.itos` (lista, índice → token) y `self.stoi` (diccionario, token → índice).
- Los dos tokens especiales, `<pad>` y `<unk>`, van **primero** en `itos`, en ese orden. Guardá sus índices en `self.pad_id` y `self.unk_id`.
- `__len__`, y `__getitem__(self, token)` que devuelve el índice del token y **nunca lanza excepción**: lo desconocido cae en `unk_id`.
- `codificar(self, texto)`: string → lista de índices.
- `decodificar(self, indices, ocultar_pad=True)`: lista de índices o tensor → lista de tokens.
- `__repr__` informativo.

Probala: construí un vocabulario con `freq_min=1` sobre `train.text` y guardalo en una variable `v`. Imprimí los primeros 12 tokens de `itos`, los índices de `<pad>` y `<unk>`, el índice de `"alarma"` y el de una palabra que no exista en el corpus.

> **Pista 1:** `collections.Counter` tiene `.most_common()`, que devuelve los pares `(token, frecuencia)` ordenados de mayor a menor. Construir `itos` a partir de ahí garantiza que los índices bajos sean las palabras frecuentes, lo cual es cómodo para inspeccionar.

> **Pista 2:** Que los especiales vayan primero no es cosmético: garantiza que `<pad>` tenga el índice 0 sin importar cómo esté configurado el resto, y del 0 dependen el tensor de ceros del Ejercicio 7 y el `padding_idx` del modelo de la Parte B.

> **Pista 3:** Para que `decodificar` acepte tanto listas como tensores, `torch.is_tensor(indices)` y `.tolist()` resuelven el caso.

In [ ]:
# Tu código aquí

**Enunciado — Parte B: cuánto pesa el umbral.**

Agregale a la clase dos métricas:

- `cobertura`, como propiedad: qué proporción de las **apariciones** del corpus con el que se construyó el vocabulario quedan cubiertas por él.
- `tasa_unk(self, textos)`: qué proporción de los tokens de un texto **nuevo** cae en `<unk>`.

Con eso, imprimí una tabla con una fila por cada valor de `freq_min` en `[1, 2, 3, 5, 10]` y cuatro columnas: el umbral, el tamaño del vocabulario que produce, su cobertura, y su tasa de `<unk>` medida sobre `test.text`. Debajo de la tabla, informá cuántos tokens del corpus aparecen **una sola vez** (los *hapax*) y qué porcentaje del vocabulario son.

> **Pista:** Podés agregarle métodos a una clase ya definida sin volver a escribirla entera: `Vocabulario.tasa_unk = tasa_unk` la agrega como método, y `Vocabulario.cobertura = property(cobertura)` la agrega como propiedad. Si preferís, escribilas adentro de la clase en la celda anterior y en esta usalas nomás.

In [ ]:
# Tu código aquí

**Enunciado — Parte C: la elección y el viaje de ida y vuelta.**

Elegí un `freq_min` que puedas justificar con los números de la tabla y construí con él el vocabulario definitivo, en una variable llamada `vocab` (es la que vas a usar en el Ejercicio 7). Imprimí su tamaño, su cobertura y su tasa de `<unk>` en prueba.

Después codificá y volvé a decodificar estas tres frases, mostrando para cada una los índices que salieron y el texto reconstruido:

```python
frases = ["pon una alarma para las siete",
          "quiero un vuelo a katmandú el jueves",
          test.text.iloc[7]]
```

> **Pista:** Codificar y decodificar la misma frase es la forma más rápida de auditar un pipeline de texto. Mirá con atención qué palabras vuelven distintas de como entraron: cada una de esas es una palabra que el modelo nunca va a ver.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Justificá el `freq_min` que elegiste, usando los números de tu tabla. ¿Qué le pasa a la tasa de `<unk>` a medida que subís el umbral, y por qué eso no es necesariamente malo?

b) La palabra `katmandú` volvió como `<unk>`. Un compañero propone arreglarlo poniendo `freq_min=1` para que ninguna palabra quede afuera. Explicá por qué eso **no** resuelve el problema de fondo, y qué le pasaría al modelo con las palabras que ve por primera vez en producción.

*(Escribí tu respuesta acá)*

### Ejercicio 7 — Truncar, rellenar y construir la máscara

**Objetivo:** Convertir un corpus de textos de largo variable en un tensor rectangular `(B, L)`, construir su máscara y medir cuánto del tensor es relleno.

**Enunciado:**

Un tensor es rectangular y las frases no lo son. Este ejercicio es el que resuelve esa tensión y produce, por fin, el tensor que la Parte B va a consumir.

1. **Escribí `codificar_lote(self, textos, largo)`**, que devuelve un tensor `(B, largo)` de tipo `torch.long` donde cada fila es un texto codificado, truncado a `largo` y rellenado con `pad_id`. Agregásela a la clase `Vocabulario` para poder llamarla como método, igual que hiciste con las métricas del ejercicio anterior.

2. **Medí el compromiso.** Para `largo` en `[8, 16, 32]`, aplicá la función sobre `train.text` completo y para cada valor imprimí: la forma del tensor, la proporción de posiciones que son relleno, y qué porcentaje de las órdenes quedaron truncadas (es decir, cuya codificación era más larga que `largo`).

3. **Elegí `L`** justificando la elección con esos números, y con ese valor construí `X_train`, el tensor de todo `train.text`.

4. **Construí la máscara** de `X_train`: el tensor booleano que vale `True` en las posiciones con token real y `False` en el relleno. Mostrá, para las primeras 5 órdenes, tres cosas: la fila de índices, la máscara como enteros, y la cantidad de tokens reales que tiene esa fila.

> **Pista 1:** Arrancá de un tensor de ceros de forma `(len(textos), largo)` y tipo `torch.long`. Como `<pad>` es el índice 0, el relleno ya viene hecho: solo hay que escribir los tokens reales encima, al principio de cada fila.

> **Pista 2:** Para el porcentaje de órdenes truncadas necesitás el largo en tokens de cada orden, antes de recortar. El método `.map()` de una serie de pandas te lo da en una línea.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Con el `L` que elegiste, más de la mitad de las posiciones del tensor son relleno. Eso significa que más de la mitad del cómputo del modelo se va a gastar procesando posiciones que no significan nada.

a) ¿Por qué, aun así, conviene armar el tensor de esta manera en vez de procesar cada orden por separado con su largo real?

b) Proponé una manera de reducir ese desperdicio **sin** bajar `L`, y explicá qué habría que cambiar en el `DataLoader` para lograrlo.

*(Escribí tu respuesta acá)*

### Ejercicio 8 — Por qué un índice no es una representación

**Objetivo:** Cerrar la parte entendiendo qué logramos con la cadena que construimos y, sobre todo, qué **no** logramos.

**Enunciado:**

Terminaste el trabajo: cualquier orden en español entra por un lado y sale como una fila de un tensor `(B, L)` de enteros, que es exactamente lo que la Parte B necesita. Pero mirá esta tabla, que sale del vocabulario que construiste, antes de festejar:

| palabra | índice | frecuencia |
|---|---|---|
| `canción` | 45 | 281 |
| `alarma` | 46 | 274 |
| `música` | 52 | 239 |
| `luz` | 81 | 134 |
| `lámpara` | 856 | 9 |
| `despertador` | 1021 | 7 |

Respondé, apoyándote en esos números:

1. `alarma` y `despertador` son casi sinónimos en este dominio y quedaron a 975 posiciones de distancia; `alarma` y `música` no tienen nada que ver y quedaron a 6. Explicá qué codifica realmente el índice, por qué la resta entre dos índices no quiere decir nada, y qué relación falsa asumiría un modelo que recibiera estos números directamente como entrada.

2. La alternativa clásica es el vector *one-hot*: largo `len(vocab)`, un 1 en la posición del token y ceros en el resto. Eso arregla el problema del punto 1, pero trae dos problemas nuevos. Nombralos —estimando, para nuestro vocabulario y con `L = 16`, cuánta memoria ocuparía un lote de 64 órdenes en *one-hot* de `float32` frente a lo que ocupa como índices— y decí qué propiedad tendría que tener una buena representación, que ni los índices ni el *one-hot* tienen.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] Los tensores que imprimo tienen las formas y los tipos que pide cada enunciado (`torch.long` para índices).
- [ ] La clase `Vocabulario` corre completa: `codificar`, `decodificar`, `cobertura`, `tasa_unk` y `codificar_lote`.
- [ ] La tabla del Ejercicio 6b tiene sus cinco filas y la elección de `freq_min` está justificada con esos números.
- [ ] Los valores numéricos que imprimo son razonables (no hay infinitos, ni `NaN`, ni proporciones fuera de `[0, 1]`).
- [ ] Respondí las ocho preguntas de análisis (Ej. 1 a 8).
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Construiste la cadena completa que va de texto en español a un tensor que una red puede consumir. En el camino practicaste:

- **Mecánica de tensores**: creación, forma, tipo, memoria compartida, *broadcasting* y `autograd`.
- **Tokenización**, y cómo medir el efecto de una decisión de diseño en vez de discutirla en abstracto.
- **La clase `Vocabulario`**, con sus tokens especiales y su umbral de frecuencia elegido con datos.
- **Truncado, relleno y máscara**, y el compromiso entre perder información y desperdiciar cómputo.

Guardá tu implementación de `Vocabulario`. La Parte B no la vuelve a pedir —su celda de setup trae una lista para usar—, pero es tu código el que va corregido en esta entrega, y conviene que compares las dos.

En la **Parte B** ese tensor entra por fin a un modelo. Vas a construir un clasificador de escenarios con una tabla de *embeddings* y un promedio enmascarado —el mismo *broadcasting* del Ejercicio 3—, entrenarlo, medirlo con una matriz de confusión y provocarle un sobreajuste a propósito para después curarlo. Y al final vas a encontrarte con el límite que motiva toda la Unidad 2: el modelo que construyas no va a poder distinguir *"apagá la luz de la cocina"* de *"la cocina apagá de luz la"*.